In [1]:
import os
import json
import google.generativeai as genai
from kscLLM.index import ROOT_PATH
from kscLLM.util import to_markdown, get_model


GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY")
print("set") if GOOGLE_API_KEY else print("unset")
genai.configure(api_key=GOOGLE_API_KEY)

set


/home/lukas/Programming/uni/threatintel-showcase/llm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = get_model()

# Load collected logs

In [3]:
with open(ROOT_PATH / "tmp/stix.json", "r") as file:
    observables_bundle = json.load(file)

prompt = f"""Report malicious activity in the following STIX Domain Objects. 
A credential access attack is searched. First a token is fetched. This token is then loaded in the application by calling the metadata.google.internal service-accounts token endpoint through a SSRF attack during image upload.
Find for each step of this attack all related bundles.

Respond in this format for each malicious SDO:

Artifact ID: <artifact id>
Message: <message of the bundle>
Explanation: <explanation why this SDO is malicious>

{str(observables_bundle)}"""


response_indicator = model.generate_content(prompt)
# to_markdown(response_indicator.text)
print(response_indicator.text)

Artifact ID: artifact--fcc56611-f6c1-5992-bca1-cac4c3054059
Message: I1007 15:55:02.559034   49221 serviceaccounts.go:168] [conn-id:8ec8719a2a94c2a3 rpc-id:beb0bc24c3350fdc remote-addr:10.1.3.9:37940 pod:users/user-service-d799868d7-psx7t] Fetched token for pod users/user-service-d799868d7-psx7t
Explanation: The message indicates that a token was fetched for a pod, which is the first step in a credential access attack.

Artifact ID: artifact--35f59223-81c1-5af6-bab0-4bb4fcaa1482
Message: I1007 15:55:02.559869   49221 metadata.go:216] [conn-id:8ec8719a2a94c2a3 rpc-id:beb0bc24c3350fdc remote-addr:10.1.3.9:37940 pod:users/user-service-d799868d7-psx7t] "/computeMetadata/v1/instance/service-accounts/default/token" HTTP/200, started at 2024-10-07 15:55:02.557574737 +0000 UTC m=+103312.383240775
Explanation: The message shows an HTTP request to the metadata service to fetch a token. This confirms the SSRF attack is successful.

Artifact ID: artifact--a30af0d0-261d-53b4-87ae-925002455abd
Messa

Search for the malicious logs:
Call to metadata endpoint: `/computeMetadata/v1/instance/service-accounts/default/token`
SSRF attack by picture upload: `metadata.google.internal`

In [4]:
true_positive_bundle_ids = [
    # "bundle--060e9e7f-1c2f-4d1c-8f60-20308669c4f3", # Fetched token for pod
    # "bundle--66c85317-a94d-495d-838a-982b82332c9b", # google.internal image upload call
 "bundle--4c0ffab6-7073-47b4-a1c2-e918a71ec245",
 "bundle--6e826f46-6c8b-481e-9304-1dea04479eb4",
]

true_positive_bundles = [bundle for bundle in observables_bundle if bundle["id"] in true_positive_bundle_ids]
true_positive_bundles

[{'type': 'bundle',
  'id': 'bundle--4c0ffab6-7073-47b4-a1c2-e918a71ec245',
  'spec_version': '2.1',
  'objects': [{'type': 'artifact',
    'spec_version': '2.1',
    'id': 'artifact--35f59223-81c1-5af6-bab0-4bb4fcaa1482',
    'mime_type': 'plain/text',
    'payload_bin': 'STEwMDcgMTU6NTU6MDIuNTU5ODY5ICAgNDkyMjEgbWV0YWRhdGEuZ286MjE2XSBbY29ubi1pZDo4ZWM4NzE5YTJhOTRjMmEzIHJwYy1pZDpiZWIwYmMyNGMzMzUwZmRjIHJlbW90ZS1hZGRyOjEwLjEuMy45OjM3OTQwIHBvZDp1c2Vycy91c2VyLXNlcnZpY2UtZDc5OTg2OGQ3LXBzeDd0XSAiL2NvbXB1dGVNZXRhZGF0YS92MS9pbnN0YW5jZS9zZXJ2aWNlLWFjY291bnRzL2RlZmF1bHQvdG9rZW4iIEhUVFAvMjAwLCBzdGFydGVkIGF0IDIwMjQtMTAtMDcgMTU6NTU6MDIuNTU3NTc0NzM3ICswMDAwIFVUQyBtPSsxMDMzMTIuMzgzMjQwNzc1',
    'defanged': False,
    'extensions': {'extension-definition--dd73de4f-a7f3-49ea-8ec1-8e884196b7a8': {'extension_type': 'toplevel-property-extension'}},
    'kubernetes': {'container_id': 'containerd://0d8b51f1a05a5786a838a171a8be982b964c377e7b5bd3ae114562479d01bacc',
     'container_image': 'gke.gcr.io/gke-met

In [5]:
with open(ROOT_PATH / "tmp/positives.json", "w") as file:
    json.dump(true_positive_bundles, file)